In [1]:
"""
Phase 1 - Step 1: Data Cleaning
Fixes two known problems in the raw ML_Engineered_Battery_Dataset.csv:
  1. Corrupted/partial batteries (B0050, B0052) that crashed mid-parse -> drop entirely
  2. Bad SoH baseline (first_capacity was a glitchy first reading) -> recompute SoH
     using the median of the first few discharge cycles instead of just the 1st one
"""
import pandas as pd
import numpy as np

RAW_PATH = "ML_Engineered_Battery_Dataset.csv"
CLEAN_PATH = "battery_dataset_clean.csv"

df = pd.read_csv(RAW_PATH)
print(f"Raw rows: {len(df)}, Raw batteries: {df['Battery_ID'].nunique()}")

# -----------------------------------------------------------------
# STEP A: Drop known corrupted/partial batteries
# -----------------------------------------------------------------
BAD_BATTERIES = ["B0050", "B0052"]  # too few rows, crashed mid-parse (see EDA discussion)
df = df[~df["Battery_ID"].isin(BAD_BATTERIES)].copy()
print(f"After dropping {BAD_BATTERIES}: {len(df)} rows, {df['Battery_ID'].nunique()} batteries")

# -----------------------------------------------------------------
# STEP B: Drop true sensor-dropout rows (Capacity_Ahr == 0)
# A discharge test that ran for real seconds cannot deliver exactly
# 0.0 Ahr - this is a sensor/logging failure, not a real measurement.
# -----------------------------------------------------------------
before = len(df)
df = df[df["Capacity_Ahr"] > 0].copy()
print(f"Dropped {before - len(df)} rows with Capacity_Ahr == 0 (sensor dropouts)")

# -----------------------------------------------------------------
# STEP C: Recompute SoH using a robust baseline capacity
# We do NOT use the first reading (or first few) as "100%", because
# some cells (e.g. B0033) show a real electrochemical "formation"
# ramp-up over their first several cycles, where measured capacity
# is artificially LOW while the electrodes are still stabilizing.
# Instead: SoH is fundamentally about fade from a battery's peak
# achievable capacity, so we use the MAXIMUM observed Capacity_Ahr
# for each battery as its "100% / nominal" baseline. This is the
# standard convention used in NASA/CALCE SoH literature.
# -----------------------------------------------------------------
baseline_capacity = df.groupby("Battery_ID")["Capacity_Ahr"].max()
baseline_capacity.name = "Baseline_Capacity_Ahr"

df = df.merge(baseline_capacity, on="Battery_ID")
df["SoH"] = df["Capacity_Ahr"] / df["Baseline_Capacity_Ahr"]

# -----------------------------------------------------------------
# STEP D: Clip SoH to [0, 1.0]
# Because baseline is now the battery's own max, SoH cannot exceed
# 1.0 by construction - this clip just guards against float noise.
# -----------------------------------------------------------------
before_clip = (df["SoH"] > 1.0).sum()
df["SoH"] = df["SoH"].clip(lower=0, upper=1.0)
print(f"Clipped {before_clip} rows where SoH > 1.0 (float rounding only)")

# -----------------------------------------------------------------
# STEP E: Drop rows with missing critical fields
# -----------------------------------------------------------------
critical_cols = ["Capacity_Ahr", "SoH", "Ambient_Temperature",
                  "Discharge_Time_Seconds", "Voltage_Drop_Rate_V_per_sec"]
before = len(df)
df = df.dropna(subset=critical_cols)
print(f"Dropped {before - len(df)} rows with missing critical values")

# -----------------------------------------------------------------
# STEP F: Sanity-check final distribution
# -----------------------------------------------------------------
print("\nFinal SoH distribution:")
print(df["SoH"].describe())

print("\nFinal battery count:", df["Battery_ID"].nunique())
print("Final row count:", len(df))

df.to_csv(CLEAN_PATH, index=False)
print(f"\nSaved cleaned dataset -> {CLEAN_PATH}")

Raw rows: 2726, Raw batteries: 32
After dropping ['B0050', 'B0052']: 2726 rows, 32 batteries
Dropped 0 rows with Capacity_Ahr == 0 (sensor dropouts)
Clipped 0 rows where SoH > 1.0 (float rounding only)
Dropped 0 rows with missing critical values

Final SoH distribution:
count    2726.000000
mean        0.757467
std         0.234085
min         0.032681
25%         0.713660
50%         0.797113
75%         0.908448
max         1.000000
Name: SoH, dtype: float64

Final battery count: 32
Final row count: 2726

Saved cleaned dataset -> battery_dataset_clean.csv


In [2]:
df

,Battery_ID,Cycle_Index,Discharge_Index,Ambient_Temperature,Discharge_Time_Seconds,Max_Temp_Reached,Min_Voltage_Recorded,Voltage_Drop_Rate_V_per_sec,Internal_Resistance_Re,Cycles_Since_Impedance,Capacity_Ahr,SoH,Baseline_Capacity_Ahr
0,B0005,2,1,24,3690.234,38.982181,2.612467,0.000428,0.044669,0,1.856487,1.000000,1.856487
1,B0005,4,2,24,3672.344,39.033398,2.587209,0.000436,0.044669,0,1.846327,0.994527,1.856487
2,B0005,6,3,24,3651.641,38.818797,2.651917,0.000421,0.044669,0,1.835349,0.988614,1.856487
3,B0005,8,4,24,3631.563,38.762305,2.592948,0.000439,0.044669,0,1.835263,0.988567,1.856487
4,B0005,10,5,24,3629.172,38.665393,2.547420,0.000452,0.044669,0,1.834646,0.988235,1.856487
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2721,B0056,241,98,4,2345.000,13.316711,2.687277,0.000637,0.103227,5,1.130219,0.840714,1.344356
2722,B0056,243,99,4,2363.047,14.612811,2.698012,0.000628,0.103227,7,1.125872,0.837481,1.344356
2723,B0056,245,100,4,2316.687,14.205466,2.692736,0.000643,0.103227,9,1.143011,0.850230,1.344356
2724,B0056,249,101,4,2322.000,13.862517,2.689910,0.000639,0.102677,1,1.137273,0.845962,1.344356


In [3]:
df['Battery_ID'].unique()

array(['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027',
       'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034',
       'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043',
       'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0049', 'B0051',
       'B0053', 'B0054', 'B0055', 'B0056'], dtype=object)